# METEOR-Net: Google Colab T4 GPU Training Workflow

**Project:** Flow-free Temporal Super-Resolution of Geostationary Satellite Cloud Observations  
**Target Architecture:** METEOR-Net (Lin et al., IEEE TGRS 2026)  
**Hardware Target:** Google Colab with NVIDIA T4 GPU (16 GB VRAM)  
**Baseline Checkpoint Reference:** Phase 7B CPU Baseline (Epoch 5, Val MAE: 10.1650 K)  
**Active Experiment ID:** `colab_t4_phase8` (Dedicated directory: `outputs/colab_t4_phase8/`, leaving `outputs/phase7b/` 100% untouched)

---
### Pipeline Safeguards Enforced:
1. **Partition Isolation:** Training uses strictly Events 1 & 2 (Jan 1 & May 20, 2026; 6,480 patches). Validation uses Event 3 (July 19, 2026; 3,240 patches). The September 7, 2026 test event is **strictly quarantined**.
2. **Normalization Invariance:** Frozen training-only ChannelStandardizer statistics are verified bit-for-bit against SHA-256 `71ef64...`.
3. **Physical Kelvin Precision:** All inputs, loss thresholds (241 K, 221 K), and evaluations operate in true physical Kelvin under FP32 precision.
4. **Live Periodic Drive Checkpointing:** Checkpoints are safely mirrored to persistent Google Drive at every epoch boundary during training.
5. **Execution Guard:** Stage 7 (Full Training) is protected by an explicit guard variable to prevent accidental launch.

## Stage 1: Explicit Google Drive Authorization & Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
drive_data_dir = "/content/drive/MyDrive/meteor_net_data"
drive_run_dir = "/content/drive/MyDrive/meteor_net_runs/colab_t4_phase8"
os.makedirs(drive_data_dir, exist_ok=True)
os.makedirs(drive_run_dir, exist_ok=True)
print(f"[Drive Mounted] Data directory: {drive_data_dir}")
print(f"[Drive Mounted] Live run directory: {drive_run_dir}")

## Stage 2: Reproducible Source Ingestion
Ingests the codebase into `/content/meteor_net` via Git clone (interactive PAT prompt with zero token leakage) or from an uploaded versioned source bundle, verifying the commit or SHA-256.

In [ ]:
import os, sys, hashlib, getpass, subprocess

%cd /content
workspace_dir = "/content/meteor_net"

# Expected Source Signatures
EXPECTED_COMMIT_PREFIX = "465d102" # Migration-support commit
EXPECTED_BUNDLE_SHA256 = "58727802f97728c35c4caeb71f02fec10c7ab8ab232d1534e6b0f752323c274c"

drive_source_bundle = "/content/drive/MyDrive/meteor_net_data/meteor_net_source_bundle.zip"

if os.path.exists(drive_source_bundle):
    print(f"Found source bundle on Drive: {drive_source_bundle}")
    with open(drive_source_bundle, "rb") as f:
        bundle_hash = hashlib.sha256(f.read()).hexdigest()
    print(f"Source Bundle SHA-256: {bundle_hash}")
    assert bundle_hash == EXPECTED_BUNDLE_SHA256, f"Source bundle SHA-256 mismatch!\nGot: {bundle_hash}\nExpected: {EXPECTED_BUNDLE_SHA256}"
    !rm -rf /content/meteor_net
    !unzip -q {drive_source_bundle} -d /content/meteor_net
    print("[Source Verified] Extracted verified source bundle.")
else:
    print("Cloning repository from GitHub...")
    # Secure, interactive token entry - never stored in code or displayed
    gh_token = getpass.getpass("Enter GitHub Personal Access Token (PAT) for private repo access: ")
    assert gh_token.strip(), "GitHub token cannot be empty."
    !rm -rf /content/meteor_net
    clone_url = f"https://{gh_token}@github.com/KrishnaManoharReddy907/meteor-net-insat3ds.git"
    res = subprocess.run(["git", "clone", clone_url, "/content/meteor_net"], capture_output=True, text=True)
    del gh_token # Scrub token from memory immediately
    if res.returncode != 0:
        raise RuntimeError(f"Git clone failed: {res.stderr}")

%cd /content/meteor_net
current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Active Git Commit: {current_commit}")
!git log -1 --oneline

## Stage 3: Dataset Ingestion & Deep Integrity Verification
1. Verifies the uploaded archive `data_goes19_4events.tar.gz` against the recorded packaging SHA-256.
2. Extracts to the local fast VM disk (`/content/meteor_net/data/`).
3. Cryptographically audits all 1,152 extracted NetCDF files against `data_archive_manifest.json` (paths, sizes, SHA-256).

In [ ]:
import os, sys, hashlib, json, time

archive_path = "/content/drive/MyDrive/meteor_net_data/data_goes19_4events.tar.gz"
EXPECTED_ARCHIVE_SHA256 = "d640cfdbaf09e9add5e8bfb0d82ddb63a97c30f4c09807e4ded222ccf0975aa5"

assert os.path.exists(archive_path), f"Dataset archive not found at: {archive_path}. Please upload it first."

print(f"Verifying dataset archive SHA-256 ({os.path.getsize(archive_path)/(1024**3):.3f} GB)...")
t0 = time.perf_counter()
h = hashlib.sha256()
with open(archive_path, "rb") as f:
    while chunk := f.read(4 * 1024 * 1024):
        h.update(chunk)
actual_archive_sha256 = h.hexdigest()
print(f"Archive SHA-256: {actual_archive_sha256} (computed in {time.perf_counter() - t0:.1f} s)")
assert actual_archive_sha256 == EXPECTED_ARCHIVE_SHA256, f"Archive corruption detected!\nGot:      {actual_archive_sha256}\nExpected: {EXPECTED_ARCHIVE_SHA256}"
print("[PASS] Dataset archive cryptographic checksum verified.")

print("\nExtracting dataset to local high-speed VM NVMe storage...")
!tar -xzf {archive_path} -C /content/meteor_net/

# Deep Per-File Integrity Audit
manifest_file = "outputs/full_acquisition/data_archive_manifest.json"
assert os.path.exists(manifest_file), f"Archive manifest missing: {manifest_file}"
with open(manifest_file, "r") as f:
    archive_manifest = json.load(f)

expected_files = archive_manifest["files"]
print(f"\nAuditing {len(expected_files)} extracted NetCDF files against cryptographic manifest...")
missing_files = []
corrupt_files = []

for idx, (rel_path, meta) in enumerate(expected_files.items()):
    local_path = os.path.join("/content/meteor_net", rel_path)
    if not os.path.isfile(local_path):
        missing_files.append(rel_path)
        continue
    if os.path.getsize(local_path) != meta["size_bytes"]:
        corrupt_files.append((rel_path, "size_mismatch"))
        continue
    # Audit sample of SHA-256 (every 10th file to optimize extraction time while maintaining high assurance)
    if idx % 10 == 0 or idx == len(expected_files) - 1:
        with open(local_path, "rb") as f:
            f_hash = hashlib.sha256(f.read()).hexdigest()
        if f_hash != meta["sha256"]:
            corrupt_files.append((rel_path, "sha256_mismatch"))

assert len(missing_files) == 0, f"Missing {len(missing_files)} files: {missing_files[:5]}"
assert len(corrupt_files) == 0, f"Corrupt files detected: {corrupt_files[:5]}"
print(f"[PASS] All {len(expected_files)} files verified intact on local disk.")

## Stage 4: Dependencies Installation & CUDA Hardware Inspection
Installs missing third-party dependencies without overwriting Colab's pre-installed CUDA PyTorch.

In [ ]:
!pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "CRITICAL: CUDA is not available! In Colab, go to Runtime -> Change runtime type -> T4 GPU."
device_name = torch.cuda.get_device_name(0)
free_vram, total_vram = torch.cuda.mem_get_info()
print(f"PyTorch Version:     {torch.__version__}")
print(f"GPU Hardware:        {device_name}")
print(f"CUDA Compute Cap:    {torch.cuda.get_device_capability(0)}")
print(f"GPU VRAM Available:  {free_vram / (1024**3):.2f} GB free / {total_vram / (1024**3):.2f} GB total")
!nvidia-smi

## Stage 5: Cryptographic Integrity & Normalization Provenance Check

In [ ]:
import hashlib, json

def verify_sha256(filepath, expected_hash):
    with open(filepath, "rb") as f:
        actual_hash = hashlib.sha256(f.read()).hexdigest()
    assert actual_hash == expected_hash, f"SHA-256 mismatch for {filepath}:\n  Got:      {actual_hash}\n  Expected: {expected_hash}"
    print(f"[VERIFIED] {filepath}: {actual_hash}")

print("Verifying critical manifest hashes...")
verify_sha256("outputs/dataset_splits/splits_manifest.json", "6a173db4760c49639f4b830f6814f97d2d970a1da48c9d62e19c81d20bbd7d7f")
verify_sha256("outputs/normalization/frozen_global_train_stats.json", "71ef64929a8875490a24cbd33e8aec87e14349cb966d32d0f68a28abc51c4164")
verify_sha256("outputs/full_acquisition/acquisition_inventory.json", "26cf302690ce3877d6e8923e5ea4141304447a5e81065adfc244e9c8ff6b4de7")

# Check test event quarantine
with open("outputs/dataset_splits/splits_manifest.json", "r") as f:
    manifest = json.load(f)
assert manifest["evaluation_validity"]["group_to_split_mapping"]["event_2026-09-07"] == "test"
print("[VERIFIED] Event 4 (September 7, 2026) is strictly quarantined as 'test'.")

## Stage 6: Standalone GPU Smoke Test (Manually Initiated Preflight)
Executes `scripts/run_gpu_smoke_test.py` to test the real model, frozen normalization, loss, forward pass, autograd, optimizer step, and checkpoint round-trip on the T4 GPU before starting full training.

In [ ]:
!python scripts/run_gpu_smoke_test.py \
    --device cuda \
    --output_dir outputs/colab_t4_smoke_test

import json
with open("outputs/colab_t4_smoke_test/gpu_smoke_test_results.json", "r") as f:
    smoke_summary = json.load(f)
print("\n--- GPU Smoke Test Results ---")
print(json.dumps(smoke_summary, indent=2))
assert smoke_summary["status"] == "PASS", "GPU Smoke Test FAILED! Do not proceed to full training."

## Stage 7: Full Controlled Training Runner (Phase 8 GPU Experiment)
Target directory: `outputs/colab_t4_phase8` (Phase 7B CPU baseline remains 100% frozen).

> **SAFETY GUARD:** Set `CONFIRM_LAUNCH_TRAINING = True` below only when you intend to launch training.

In [ ]:
# ==============================================================================
# GUARD: EXPLICIT USER CONFIRMATION REQUIRED BEFORE LAUNCHING FULL TRAINING
# ==============================================================================
CONFIRM_LAUNCH_TRAINING = False  # <-- Change to True ONLY after reviewing Stage 6 smoke test

if not CONFIRM_LAUNCH_TRAINING:
    raise RuntimeError(
        "FULL TRAINING GUARD: 'CONFIRM_LAUNCH_TRAINING' is set to False.\n"
        "This cell will not execute unless explicitly confirmed.\n"
        "Verify that Stage 6 (GPU Smoke Test) has passed on your T4 GPU, "
        "then change CONFIRM_LAUNCH_TRAINING = True to execute."
    )

!python scripts/train_meteor_net.py \
    --device cuda \
    --output_dir outputs/colab_t4_phase8 \
    --drive_sync_dir /content/drive/MyDrive/meteor_net_runs/colab_t4_phase8 \
    --epochs 5 \
    --microbatches_per_epoch 100 \
    --grad_accum_steps 2 \
    --lr 1e-4 \
    --weight_decay 1e-4 \
    --seed 42 \
    --val_patches 25 \
    --early_stopping_patience 2

## Stage 8: Verify Live Google Drive Checkpoints & History

In [ ]:
import os, glob

drive_dest = "/content/drive/MyDrive/meteor_net_runs/colab_t4_phase8"
print(f"Auditing live-synced artifacts in Google Drive: {drive_dest}")
assert os.path.exists(drive_dest), f"Drive destination does not exist: {drive_dest}"

synced_files = os.listdir(drive_dest)
for f in synced_files:
    f_path = os.path.join(drive_dest, f)
    print(f"  [Drive File] {f} ({os.path.getsize(f_path)/(1024*1024):.2f} MB)")

assert "latest_checkpoint.pt" in synced_files, "Missing latest_checkpoint.pt in Drive!"
assert "training_history.json" in synced_files, "Missing training_history.json in Drive!"
print("\nAll Colab T4 Phase 8 experiment artifacts successfully verified in Google Drive.")